In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("D:\\Deep_Learning\\Dataset\\diabetes.csv")

In [4]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [6]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [7]:
from sklearn.model_selection import  train_test_split
from sklearn.preprocessing import StandardScaler

In [8]:
scaler = StandardScaler()

In [9]:
X = scaler.fit_transform(X)

In [10]:
X.shape

(768, 8)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 32)

In [48]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense
from keras.layers import Dropout

In [13]:
model = Sequential()
model.add(Dense(32, activation = 'relu', input_dim = 8))
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

d:\Deep_Learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(optimizer = 'Adam', loss = "binary_crossentropy", metrics = ["accuracy"])

In [15]:
model.fit(X_train, y_train, batch_size = 32, epochs = 10,validation_data = (X_test, y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4951 - loss: 0.8127 - val_accuracy: 0.5909 - val_loss: 0.7297
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6107 - loss: 0.7127 - val_accuracy: 0.6623 - val_loss: 0.6458
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6612 - loss: 0.6442 - val_accuracy: 0.7078 - val_loss: 0.5924
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6954 - loss: 0.5989 - val_accuracy: 0.7338 - val_loss: 0.5546
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7117 - loss: 0.5672 - val_accuracy: 0.7273 - val_loss: 0.5284
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7166 - loss: 0.5442 - val_accuracy: 0.7403 - val_loss: 0.5089
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7248 - loss: 0.5274 - val_accuracy: 0.7468 - val_loss: 0.4939
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7296 - loss: 0.5147 - val_accuracy: 0.7532 - val_loss:

### 1. How to select appropriate optimizer
### 2. No of nodes in a layer
### 3. How to select no. of layers
### 4. All in all one model

In [16]:
import kerastuner as kt

C:\Users\CSN\AppData\Local\Temp\ipykernel_1960\1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [17]:
def build_model(hp):
    model = Sequential()

    model.add(Dense(32, activation = 'relu', input_dim = 8))
    model.add(Dense(1, activation = 'sigmoid'))

    optimizer = hp.Choice('optimizer', values = ['adam', 'sgd', 'rmsprop', 'adadelta'])

    model.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics =["accuracy"])

    return model

In [18]:
tuner = kt.RandomSearch(build_model, 
                        objective = 'val_accuracy',
                        max_trials = 5)

Reloading Tuner from .\untitled_project\tuner0.json


In [19]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

In [20]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [21]:
model = tuner.get_best_models(num_models=1)[0]

d:\Deep_Learning\venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.fit(X_train, y_train, batch_size = 32, epochs=100, initial_epoch= 6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7313 - loss: 0.5930 - val_accuracy: 0.7922 - val_loss: 0.5266
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7459 - loss: 0.5627 - val_accuracy: 0.7987 - val_loss: 0.5014
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7476 - loss: 0.5414 - val_accuracy: 0.8117 - val_loss: 0.4835
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7524 - loss: 0.5254 - val_accuracy: 0.8052 - val_loss: 0.4716
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7590 - loss: 0.5143 - val_accuracy: 0.7987 - val_loss: 0.4617
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7655 - loss: 0.5050 - val_accuracy: 0.7857 - val_loss: 0.4550
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7622 - loss: 0.4982 - val_accuracy: 0.7727 - val_loss: 0.4505
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7687 - loss: 0.4926 - val_accuracy: 0.779

In [24]:
def build_model(hp):
    model = Sequential()

    units = hp.Int('Units', min_value = 8, max_value = 128, )

    model.add(Dense(units = units, activation = 'relu', input_dim = 8))
    model.add(Dense(1, activation= 'sigmoid'))

    model.compile(optimizer='adam', loss = 'binary_crossentropy', metrics=['accuracy'])

    return model

In [25]:
tuner = kt.RandomSearch(build_model, 
                        objective = 'val_accuracy',
                        max_trials = 5, 
                        directory = 'mydir',
                        project_name = 'No. of Neurons')

Reloading Tuner from mydir\No. of Neurons\tuner0.json


In [26]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

In [27]:
tuner.get_best_hyperparameters()[0].values

{'Units': 120}

In [28]:
model = tuner.get_best_models(num_models = 1)[0]

In [29]:
model.fit(X_train, y_train, batch_size = 32, epochs = 100, initial_epoch=6, validation_data = (X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7720 - loss: 0.4870 - val_accuracy: 0.7857 - val_loss: 0.4407
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7752 - loss: 0.4763 - val_accuracy: 0.8117 - val_loss: 0.4362
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7752 - loss: 0.4699 - val_accuracy: 0.8117 - val_loss: 0.4317
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7769 - loss: 0.4654 - val_accuracy: 0.8117 - val_loss: 0.4281
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7785 - loss: 0.4621 - val_accuracy: 0.8182 - val_loss: 0.4248
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7818 - loss: 0.4582 - val_accuracy: 0.8182 - val_loss: 0.4243
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7932 - loss: 0.4551 - val_accuracy: 0.8182 - val_loss: 0.4261
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7915 - loss: 0.4528 - val_accuracy: 0.818

In [30]:
def build_model(hp):
    model = Sequential()

    model.add(Dense(120, activation = 'relu',input_dim = 8))

    for i in range(hp.Int('num_layers', min_value = 1, max_value = 10)):
         model.add(Dense(120, activation='relu'))

    model.add(Dense(1, activation= "sigmoid"))
    
    model.compile(optimizer='adam', loss = "binary_crossentropy", metrics = ["accuracy"])
    return model

In [31]:
tuner = kt.RandomSearch(build_model, 
                        objective= 'val_accuracy',
                        max_trials = 3,
                        directory = 'mydir',
                        project_name = 'num_layers')

Reloading Tuner from mydir\num_layers\tuner0.json


In [32]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

In [33]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 5}

In [34]:
model = tuner.get_best_models(num_models = 1)[0]

d:\Deep_Learning\venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 30 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [35]:
model.fit(X_train, y_train, epochs = 40, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7785 - loss: 0.4804 - val_accuracy: 0.7922 - val_loss: 0.4511
Epoch 8/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7932 - loss: 0.4511 - val_accuracy: 0.7922 - val_loss: 0.4371
Epoch 9/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7915 - loss: 0.4398 - val_accuracy: 0.8052 - val_loss: 0.4584
Epoch 10/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7899 - loss: 0.4359 - val_accuracy: 0.7922 - val_loss: 0.4628
Epoch 11/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7997 - loss: 0.4242 - val_accuracy: 0.7922 - val_loss: 0.4939
Epoch 12/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8078 - loss: 0.3997 - val_accuracy: 0.7857 - val_loss: 0.5085
Epoch 13/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8274 - loss: 0.4063 - val_accuracy: 0.7857 - val_loss: 0.4709
Epoch 14/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8436 - loss: 0.3752 - val_accuracy: 0.7727 - val_

In [56]:
def build_model(hp):

    model = Sequential()

    counter = 0

    for i in range(hp.Int('num_layers', min_value = 1, max_value = 10)):

        if counter == 0:
            model.add(
                Dense(
                    hp.Int("units" + str(i), min_value = 8, max_value = 128, step = 8),
                    activation = hp.Choice('activaion' + str(i), values = ["relu", 'tanh', 'sigmoid']),
                    input_dim = 8
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i), values = [0.1,0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))
        else:
            model.add(
                Dense(
                    hp.Int("units" + str(i), min_value = 8, max_value = 128, step = 8),
                    activation = hp.Choice('activaion' + str(i), values = ["relu", 'tanh', 'sigmoid'])
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i), values = [0.1,0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))
        counter += 1

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'] ), loss = "binary_crossentropy", metrics = ["accuracy"])

    return model

In [57]:
tuner = kt.RandomSearch(build_model,
                        objective= "val_accuracy",
                        max_trials = 3,
                        directory = "mydir",
                        project_name = "all_hyperparameters")

Reloading Tuner from mydir\all_hyperparameters\tuner0.json


In [58]:
tuner.search(X_train,y_train, epochs = 5, validation_data = (X_test, y_test))

In [62]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10,
 'units0': 128,
 'activaion0': 'tanh',
 'optimizer': 'rmsprop',
 'units1': 40,
 'activaion1': 'relu',
 'units2': 120,
 'activaion2': 'relu',
 'units3': 104,
 'activaion3': 'sigmoid',
 'units4': 8,
 'activaion4': 'relu',
 'units5': 8,
 'activaion5': 'relu',
 'units6': 8,
 'activaion6': 'relu',
 'units7': 8,
 'activaion7': 'relu',
 'units8': 8,
 'activaion8': 'relu',
 'units9': 8,
 'activaion9': 'relu',
 'dropout0': 0.1,
 'dropout1': 0.1,
 'dropout2': 0.1,
 'dropout3': 0.1,
 'dropout4': 0.1,
 'dropout5': 0.1,
 'dropout6': 0.1,
 'dropout7': 0.1,
 'dropout8': 0.1,
 'dropout9': 0.1}

In [61]:
model  = tuner.get_best_models(num_models=1)[0]

In [63]:
model.fit(X_train, y_train, epochs = 200, initial_epoch= 5,validation_data=(X_test, y_test) )

Epoch 6/200


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7020 - loss: 0.5954 - val_accuracy: 0.6429 - val_loss: 0.5063
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6466 - loss: 0.5566 - val_accuracy: 0.6429 - val_loss: 0.5108
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6580 - loss: 0.5733 - val_accuracy: 0.6429 - val_loss: 0.5022
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6775 - loss: 0.5585 - val_accuracy: 0.6429 - val_loss: 0.4940
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6531 - loss: 0.5555 - val_accuracy: 0.6429 - val_loss: 0.4934
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6466 - loss: 0.5317 - val_accuracy: 0.6429 - val_loss: 0.4863
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6531 - loss: 0.5584 - val_accuracy: 0.6429 - val_loss: 0.4898
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6726 - loss: 0.5391 - val_accuracy: 0.7922 - val_loss: